# 07 · Neural networks and text analytics (big data projects)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/07_neural_nets_and_text.ipynb)

**Goal:** train a small neural network, see how its hyperparameters matter, then run a complete text
project on financial headlines: tokenise, build a document-term matrix, TF-IDF, and classify sentiment.

In [1]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

Ready. Data folder: ../data/


## 1. A neural network (multi-layer perceptron) for default prediction

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import warnings; warnings.filterwarnings("ignore")

cr = pd.read_csv(DATA + "credit_default.csv")
X = pd.get_dummies(cr.drop(columns="default"), columns=["purpose"], drop_first=True, dtype=float)
y = cr["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

for layers in [(4,), (16,), (64, 64)]:
    for lr in [0.001, 0.01]:
        nn = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=layers, learning_rate_init=lr,
                                                           max_iter=600, random_state=0)).fit(X_train, y_train)
        print(f"layers {str(layers):9} lr {lr:<6} train AUC {roc_auc_score(y_train, nn.predict_proba(X_train)[:,1]):.3f}"
              f"  test AUC {roc_auc_score(y_test, nn.predict_proba(X_test)[:,1]):.3f}")

layers (4,)      lr 0.001  train AUC 0.841  test AUC 0.821
layers (4,)      lr 0.01   train AUC 0.861  test AUC 0.806


layers (16,)     lr 0.001  train AUC 0.884  test AUC 0.818


layers (16,)     lr 0.01   train AUC 0.960  test AUC 0.778


layers (64, 64)  lr 0.001  train AUC 1.000  test AUC 0.785


layers (64, 64)  lr 0.01   train AUC 1.000  test AUC 0.781


Hyperparameters (layers, nodes, learning rate, epochs) are set by **you**. Bigger nets fit the training data better
but can generalise worse: the same bias–variance trade-off as before.

## 2. What one neuron does: weighted sum + activation

In [3]:
x_in = np.array([0.5, -1.2, 2.0]); w = np.array([0.8, 0.1, -0.4]); b = 0.2
z = x_in @ w + b
print(f"summation z = {z:.3f} | ReLU = {max(0, z):.3f} | sigmoid = {1/(1+np.exp(-z)):.3f}")

summation z = -0.320 | ReLU = 0.000 | sigmoid = 0.421


## 3. Text project: from raw headlines to features

In [4]:
import re
from collections import Counter
hl = pd.read_csv(DATA + "headlines.csv")
print(hl["sentiment"].value_counts()); hl.head()

sentiment
positive    116
negative    110
neutral      74
Name: count, dtype: int64


,headline,sentiment
0,Wayne Enterprises margin expansion on solid qu...,positive
1,Hooli schedules earnings date in New York,neutral
2,Initech strong demand amid robust orders,neutral
3,Acme Corp downgraded to sell after weak quarter,negative
4,Umbrella Inc profit surges after strong holida...,positive


In [5]:
STOP = {"as", "on", "to", "in", "for", "the", "a", "this", "after", "amid", "inc", "corp"}
def tokenize(s):
    tokens = re.findall(r"[a-z]+", s.lower())        # lowercase + strip punctuation/numbers
    return [t for t in tokens if t not in STOP]       # remove stop words
def crude_stem(t):
    for suf in ("ing", "es", "ed", "s"):
        if t.endswith(suf) and len(t) > len(suf) + 2:
            return t[: -len(suf)]
    return t
ex = hl["headline"].iloc[0]
print(ex); print("tokens:", tokenize(ex)); print("stems: ", [crude_stem(t) for t in tokenize(ex)])
print(Counter(t for h in hl["headline"] for t in tokenize(h)).most_common(12))

Wayne Enterprises margin expansion on solid quarter
tokens: ['wayne', 'enterprises', 'margin', 'expansion', 'solid', 'quarter']
stems:  ['wayne', 'enterpris', 'margin', 'expansion', 'solid', 'quarter']
[('quarter', 73), ('sales', 64), ('weak', 54), ('globex', 45), ('strong', 43), ('umbrella', 41), ('stark', 41), ('industries', 41), ('hooli', 40), ('wayne', 39), ('enterprises', 39), ('cloud', 36)]


## 4. Bag-of-words, n-grams, document-term matrix and TF-IDF with scikit-learn

In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
bow = CountVectorizer(ngram_range=(1, 2), stop_words="english")
dtm = bow.fit_transform(hl["headline"])
print("Document-term matrix:", dtm.shape, "(documents × tokens incl. bigrams)")
pd.DataFrame(dtm[:3].toarray(), columns=bow.get_feature_names_out()).loc[:, lambda d: d.sum() > 0]

Document-term matrix: (300, 380) (documents × tokens incl. bigrams)


,amid,amid robust,date,date new,demand,demand amid,earnings,earnings date,enterprises,enterprises margin,...,robust orders,schedules,schedules earnings,solid,solid quarter,strong,strong demand,wayne,wayne enterprises,york
0,0,0,0,0,0,0,0,0,1,1,...,0,0,0,1,1,0,0,1,1,0
1,0,0,1,1,0,0,1,1,0,0,...,0,1,1,0,0,0,0,0,0,1
2,1,1,0,0,1,1,0,0,0,0,...,1,0,0,0,0,1,1,0,0,0


In [7]:
Xtr, Xte, ytr, yte = train_test_split(hl["headline"], hl["sentiment"], test_size=0.3, random_state=0, stratify=hl["sentiment"])
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(classification_report(yte, pred, digits=3))
print(pd.DataFrame(confusion_matrix(yte, pred, labels=clf.classes_), index=clf.classes_, columns=clf.classes_))

              precision    recall  f1-score   support

    negative      0.941     0.970     0.955        33
     neutral      1.000     0.909     0.952        22
    positive      0.944     0.971     0.958        35

    accuracy                          0.956        90
   macro avg      0.962     0.950     0.955        90
weighted avg      0.957     0.956     0.956        90

          negative  neutral  positive
negative        32        0         1
neutral          1       20         1
positive         1        0        34


In [8]:
# Most informative words per class
vec, lr = clf[0], clf[-1]
names = vec.get_feature_names_out()
for i, c in enumerate(lr.classes_):
    print(c.ljust(8), ", ".join(names[np.argsort(lr.coef_[i])[-6:]][::-1]))
print(clf.predict(["Initech beats estimates as demand grows", "Globex cuts guidance after weak quarter"]))

negative weak, slowing, slowing sales, amid slowing, after weak, weak quarter
neutral  week, this, this week, new, for, for next
positive on solid, solid quarter, solid, strong, after strong, season
['positive' 'negative']


### Exercise 1

Replace the logistic regression with `MultinomialNB()` (naive Bayes) and `CountVectorizer(ngram_range=(1, 2))`. Store its macro F1 on the test set in `nb_f1` and compare with the logistic model.

In [9]:
# Your code here

In [10]:
check("07.1");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 07.1: not answered yet. Store your answer in `nb_f1`.


### Exercise 2

Write three headlines of your own and classify them. Can you fool the model?

In [11]:
# Your code here

### Exercise 3

Train the (64, 64) neural network (`random_state=0`, with StandardScaler) for only `max_iter=20`. Store its test AUC in `auc_test_20`. What happens to train and test AUC compared with 600 iterations?

In [12]:
# Your code here

In [13]:
check("07.3");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 07.3: not answered yet. Store your answer in `auc_test_20`.


---
## Solutions

In [14]:
from sklearn.metrics import f1_score
nb = make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB()).fit(Xtr, ytr)
nb_f1 = f1_score(yte, nb.predict(Xte), average="macro")
print("1) NB macro F1:", round(nb_f1, 3), "| logistic:", round(f1_score(yte, pred, average="macro"), 3))
print("2)", clf.predict(["Hooli loses key customer but raises guidance", "Vandelay schedules earnings date", "Acme record revenue"]))
nn = make_pipeline(StandardScaler(), MLPClassifier((64, 64), max_iter=20, random_state=0)).fit(X_train, y_train)
auc_test_20 = roc_auc_score(y_test, nn.predict_proba(X_test)[:, 1])
print("3) train AUC", round(roc_auc_score(y_train, nn.predict_proba(X_train)[:, 1]), 3), "test AUC", round(auc_test_20, 3))
check_all("07")

1) NB macro F1: 0.955 | logistic: 0.955
2) ['negative' 'neutral' 'positive']


3) train AUC 0.85 test AUC 0.821
✅ Exercise 07.1: correct.


✅ Exercise 07.3: correct.

2 of 2 exercises correct.
